In [1]:
# ============================================================================
# CELL 1: Setup - Path Resolution & Imports
# ============================================================================

from pathlib import Path
import sys
import time
import logging

# Quiet mode - we only want timing results
logging.getLogger().setLevel(logging.ERROR)

# Find ModelPipeline root
current = Path.cwd()
model_root = None
for parent in [current] + list(current.parents):
    if parent.name == "ModelPipeline":
        model_root = parent
        break

if model_root is None:
    raise RuntimeError("Cannot find 'ModelPipeline' root in path tree")

if str(model_root) not in sys.path:
    sys.path.insert(0, str(model_root))

print(f"✓ ModelPipeline root: {model_root}\n")

✓ ModelPipeline root: d:\JoelDesktop folds_24\NEU FALL2025\MLops IE7374 Project\FinSights\ModelPipeline



In [2]:
# ============================================================================
# CELL 2: Full Pipeline Latency Tracking
# ============================================================================

from finrag_ml_tg1.rag_modules_src.synthesis_pipeline.supply_lines import (
    init_rag_components,
    run_supply_line_1_kpi,
    run_supply_line_2_rag,
)
from finrag_ml_tg1.rag_modules_src.prompts.prompt_loader import PromptLoader
from finrag_ml_tg1.rag_modules_src.synthesis_pipeline.bedrock_client import (
    create_bedrock_client_from_config
)
from finrag_ml_tg1.loaders.ml_config_loader import MLConfig

# ════════════════════════════════════════════════════════════════════════════
# Test Query (Complex, Multi-Company, Multi-Year, Multi-Metric)
# ════════════════════════════════════════════════════════════════════════════



# query = """From 2016 to 2022, show me financial performance for Radian Group, Exxon Mobil, 
# Netflix, Costco, and Mastercard. I need: Revenue, net income, operating cash flow, gross profit, 
# return on assets, debt-to-assets ratio, and stockholders' equity for each company by year."""

# query = """What were Tesla's total revenue and net income for fiscal year 2021?"""

# query = """How did Amazon and Meta describe their AI strategy and competitive positioning 
# in their 2022 MD&A sections? What were their main concerns about regulatory risks and 
# market competition from cloud providers and social media platforms?"""

query = """Compare Amazon, Meta, Google, and Tesla from 2018 to 2022: 
(1) Revenue growth, operating margins, and R&D spending trends
(2) How each company's management discussed AI/ML investments in their MD&A
(3) Regulatory and antitrust risks mentioned in Risk Factors sections
(4) Supply chain vulnerabilities and competitive moats described in their strategic positioning

Show both quantitative metrics and qualitative explanations for how these companies evolved 
their tech platform strategies during this period."""

print("="*80)
print("LATENCY PROFILING - FinRAG Pipeline")
print("="*80)
print(f"Query: {query[:80]}...")
print("="*80 + "\n")

# Storage for all timing measurements (in milliseconds)
timings = {}

# ════════════════════════════════════════════════════════════════════════════
# STAGE 1: Initialization (Config + Components)
# ════════════════════════════════════════════════════════════════════════════

t0 = time.perf_counter()
config = MLConfig()
rag = init_rag_components()
prompt_loader = PromptLoader()
llm_client = create_bedrock_client_from_config(config, model_key=None)
timings['1_initialization'] = (time.perf_counter() - t0) * 1000

print(f"✓ Stage 1: Initialization complete ({timings['1_initialization']:.0f}ms)")

# ════════════════════════════════════════════════════════════════════════════
# STAGE 2: KPI Pipeline (Entity Extraction + Metric Lookup)
# ════════════════════════════════════════════════════════════════════════════

t0 = time.perf_counter()
kpi_block, kpi_entities, metric_result = run_supply_line_1_kpi(query, rag)
timings['2_kpi_pipeline'] = (time.perf_counter() - t0) * 1000

print(f"✓ Stage 2: KPI Pipeline complete ({timings['2_kpi_pipeline']:.0f}ms)")

# ════════════════════════════════════════════════════════════════════════════
# STAGE 3: RAG Pipeline (Semantic Search + Expansion + Assembly)
# ════════════════════════════════════════════════════════════════════════════

t0 = time.perf_counter()
rag_block, rag_entities, bundle, unique_sents, context_str = run_supply_line_2_rag(query, rag)
timings['3_rag_pipeline'] = (time.perf_counter() - t0) * 1000

print(f"✓ Stage 3: RAG Pipeline complete ({timings['3_rag_pipeline']:.0f}ms)")

# ════════════════════════════════════════════════════════════════════════════
# STAGE 4: Context Assembly (Combine KPI + RAG + Query Footer)
# ════════════════════════════════════════════════════════════════════════════

t0 = time.perf_counter()
footer = f"\n\n{'='*70}\nUSER QUESTION\n{'='*70}\n\n{query}"
combined_context = kpi_block + "\n\n" + rag_block + footer
timings['4_context_combine'] = (time.perf_counter() - t0) * 1000

print(f"✓ Stage 4: Context assembly complete ({timings['4_context_combine']:.0f}ms)")

# ════════════════════════════════════════════════════════════════════════════
# STAGE 5: Prompt Formatting (Load System + Format User)
# ════════════════════════════════════════════════════════════════════════════

t0 = time.perf_counter()
system_prompt = prompt_loader.load_system_prompt()
user_prompt = prompt_loader.format_query_template(combined_context)
timings['5_prompt_format'] = (time.perf_counter() - t0) * 1000

print(f"✓ Stage 5: Prompt formatting complete ({timings['5_prompt_format']:.0f}ms)")

# ════════════════════════════════════════════════════════════════════════════
# STAGE 6: LLM Synthesis (AWS Bedrock API Call)
# ════════════════════════════════════════════════════════════════════════════

t0 = time.perf_counter()
llm_response = llm_client.invoke(system=system_prompt, user=user_prompt)
timings['6_llm_synthesis'] = (time.perf_counter() - t0) * 1000

print(f"✓ Stage 6: LLM synthesis complete ({timings['6_llm_synthesis']:.0f}ms)")

# ════════════════════════════════════════════════════════════════════════════
# Calculate Total
# ════════════════════════════════════════════════════════════════════════════

timings['TOTAL'] = sum(timings.values())

# ════════════════════════════════════════════════════════════════════════════
# RESULTS: Latency Breakdown Table
# ════════════════════════════════════════════════════════════════════════════

print("\n" + "="*80)
print("LATENCY BREAKDOWN")
print("="*80)
print(f"{'Stage':<40} {'Time (ms)':>12} {'Seconds':>10} {'% Total':>12}")
print("-"*80)

for stage, ms in timings.items():
    if stage != 'TOTAL':
        sec = ms / 1000
        pct = (ms / timings['TOTAL']) * 100
        stage_name = stage.split('_', 1)[1].replace('_', ' ').title()
        print(f"{stage_name:<40} {ms:>12.0f} {sec:>10.2f}s {pct:>11.1f}%")

print("-"*80)
print(f"{'TOTAL END-TO-END':<40} {timings['TOTAL']:>12.0f} {timings['TOTAL']/1000:>10.2f}s {'100.0%':>12}")
print("="*80)

# ════════════════════════════════════════════════════════════════════════════
# RESULTS: Key Metrics
# ════════════════════════════════════════════════════════════════════════════

print(f"\n[Context Statistics]")
print(f"  Combined context: {len(combined_context):,} chars (~{len(combined_context)//4:,} tokens)")
print(f"  KPI block: {len(kpi_block):,} chars")
print(f"  RAG block: {len(rag_block):,} chars")

print(f"[LLM Invocation]")
print(f"  Model: {llm_response['model_id']}")
print(f"  Input tokens: {llm_response['usage']['input_tokens']:,}")
print(f"  Output tokens: {llm_response['usage']['output_tokens']:,}")
total_tokens = llm_response['usage']['input_tokens'] + llm_response['usage']['output_tokens']
print(f"  Total tokens: {total_tokens:,}")
print(f"  Cost: ${llm_response['cost']:.4f}")

print(f"\n[Latency Estimates]")
total_sec = timings['TOTAL'] / 1000
print(f"  Observed (P50): {total_sec:.1f}s")
print(f"  Estimated P95: {total_sec * 1.15:.1f}s  (observed + 15% network variance)")
print(f"  Estimated P99: {total_sec * 1.30:.1f}s  (observed + 30% tail latency)")

print(f"\n[Bottleneck Analysis]")
llm_pct = (timings['6_llm_synthesis'] / timings['TOTAL']) * 100
pipeline_pct = 100 - llm_pct
print(f"  LLM synthesis: {llm_pct:.1f}% of total latency")
print(f"  Pipeline overhead: {pipeline_pct:.1f}% of total latency")
print(f"  → Primary bottleneck: {'LLM' if llm_pct > 70 else 'Pipeline'}")

print("\n" + "="*80)
print("✓ PROFILING COMPLETE")
print("="*80)


LATENCY PROFILING - FinRAG Pipeline
Query: Compare Amazon, Meta, Google, and Tesla from 2018 to 2022: 
(1) Revenue growth, ...

[DEBUG] ✓ Found ModelPipeline via file path: D:\JoelDesktop folds_24\NEU FALL2025\MLops IE7374 Project\FinSights\ModelPipeline
[DEBUG] ✓ AWS credentials loaded from aws_credentials.env
[DEBUG] ✓ Found ModelPipeline via file path: D:\JoelDesktop folds_24\NEU FALL2025\MLops IE7374 Project\FinSights\ModelPipeline
[DEBUG] ✓ AWS credentials loaded from aws_credentials.env
[DEBUG] ✓ Local dev detected → LOCAL_CACHE mode
[DEBUG] ✓ Found ModelPipeline via file path: D:\JoelDesktop folds_24\NEU FALL2025\MLops IE7374 Project\FinSights\ModelPipeline
[DEBUG] ✓ AWS credentials loaded from aws_credentials.env
✓ Stage 1: Initialization complete (493ms)
✓ Stage 2: KPI Pipeline complete (924ms)
✓ Stage 3: RAG Pipeline complete (6383ms)
✓ Stage 4: Context assembly complete (0ms)
✓ Stage 5: Prompt formatting complete (0ms)
[DEBUG] Raw length: 25435
[DEBUG] Cleaned length: 25373


### Query Wise Logs: 1-3.

----
================================================================================
query = """From 2016 to 2022, show me financial performance for Radian Group, Exxon Mobil, Netflix, Costco, and Mastercard. I need: Revenue, net income, operating cash flow, gross profit, return on assets, debt-to-assets ratio, and stockholders' equity for each company by year."""
================================================================================

```python
✓ Stage 1: Initialization complete (619ms)
✓ Stage 2: KPI Pipeline complete (452ms)
✓ Stage 3: RAG Pipeline complete (6261ms)
✓ Stage 4: Context assembly complete (0ms)
✓ Stage 5: Prompt formatting complete (0ms)
[DEBUG] Raw length: 11447
[DEBUG] Cleaned length: 11411
[DEBUG] First 200 chars: # Financial Performance Analysis: Five-Company Comparison (2016-2022)

The provided dataset contains comprehensive financial metrics across the five companies for the 2016-2022 period, though coverage
✓ Stage 6: LLM synthesis complete (29460ms)

================================================================================
LATENCY BREAKDOWN
================================================================================
Stage                                       Time (ms)    Seconds      % Total
--------------------------------------------------------------------------------
Initialization                                    619       0.62s         1.7%
Kpi Pipeline                                      452       0.45s         1.2%
Rag Pipeline                                     6261       6.26s        17.0%
Context Combine                                     0       0.00s         0.0%
Prompt Format                                       0       0.00s         0.0%
Llm Synthesis                                   29460      29.46s        80.1%
--------------------------------------------------------------------------------
TOTAL END-TO-END                                36792      36.79s       100.0%
================================================================================

[Context Statistics]
  Combined context: 45,079 chars (~11,269 tokens)
  KPI block: 3,394 chars
  RAG block: 41,255 chars
[LLM Invocation]
  Model: us.anthropic.claude-haiku-4-5-20251001-v1:0
  Input tokens: 16,469
  Output tokens: 3,465
  Total tokens: 19,934
  Cost: $0.0338

[Latency Estimates]
  Observed (P50): 36.8s
  Estimated P95: 42.3s  (observed + 15% network variance)
  Estimated P99: 47.8s  (observed + 30% tail latency)

[Bottleneck Analysis]
  LLM synthesis: 80.1% of total latency
  Pipeline overhead: 19.9% of total latency
  → Primary bottleneck: LLM
```

----
----

================================================================================
Query: What were Tesla's total revenue and net income for fiscal year 2021?...
================================================================================

```python
✓ Stage 1: Initialization complete (378ms)
✓ Stage 2: KPI Pipeline complete (61ms)
✓ Stage 3: RAG Pipeline complete (5796ms)
✓ Stage 4: Context assembly complete (0ms)
✓ Stage 5: Prompt formatting complete (0ms)
[DEBUG] Raw length: 2346
[DEBUG] Cleaned length: 2340
[DEBUG] First 200 chars: Tesla's fiscal year 2021 financial performance demonstrated substantial growth across both top-line and bottom-line metrics. The company recognized total revenues of USD 53.82 billion in 2021, represe
✓ Stage 6: LLM synthesis complete (9553ms)

================================================================================
LATENCY BREAKDOWN
================================================================================
Stage                                       Time (ms)    Seconds      % Total
--------------------------------------------------------------------------------
Initialization                                    378       0.38s         2.4%
Kpi Pipeline                                       61       0.06s         0.4%
Rag Pipeline                                     5796       5.80s        36.7%
Context Combine                                     0       0.00s         0.0%
Prompt Format                                       0       0.00s         0.0%
Llm Synthesis                                    9553       9.55s        60.5%
--------------------------------------------------------------------------------
TOTAL END-TO-END                                15788      15.79s       100.0%
================================================================================

[Context Statistics]
  Combined context: 32,616 chars (~8,154 tokens)
  KPI block: 467 chars
  RAG block: 31,920 chars
[LLM Invocation]
  Model: us.anthropic.claude-haiku-4-5-20251001-v1:0
  Input tokens: 12,107
  Output tokens: 603
  Total tokens: 12,710
  Cost: $0.0151

[Latency Estimates]
  Observed (P50): 15.8s
  Estimated P95: 18.2s  (observed + 15% network variance)
  Estimated P99: 20.5s  (observed + 30% tail latency)

[Bottleneck Analysis]
  LLM synthesis: 60.5% of total latency
  Pipeline overhead: 39.5% of total latency
  → Primary bottleneck: Pipeline

================================================================================
✓ PROFILING COMPLETE
================================================================================
```

----
----

================================================================================
query = """How did Amazon and Meta describe their AI strategy and competitive positioning 
in their 2022 MD&A sections? What were their main concerns about regulatory risks and 
market competition from cloud providers and social media platforms?"""
================================================================================

```python
✓ Stage 1: Initialization complete (386ms)
✓ Stage 2: KPI Pipeline complete (594ms)
✓ Stage 3: RAG Pipeline complete (6989ms)
✓ Stage 4: Context assembly complete (0ms)
✓ Stage 5: Prompt formatting complete (0ms)
[DEBUG] Raw length: 8954
[DEBUG] Cleaned length: 8936
[DEBUG] First 200 chars: # Analysis of Amazon and Meta's 2022 AI Strategy, Competitive Positioning, and Risk Factors

The provided dataset contains substantial narrative context from Amazon and Meta's 2022 10-K filings regard
✓ Stage 6: LLM synthesis complete (23061ms)

================================================================================
LATENCY BREAKDOWN
================================================================================
Stage                                       Time (ms)    Seconds      % Total
--------------------------------------------------------------------------------
Initialization                                    386       0.39s         1.2%
Kpi Pipeline                                      594       0.59s         1.9%
Rag Pipeline                                     6989       6.99s        22.5%
Context Combine                                     0       0.00s         0.0%
Prompt Format                                       0       0.00s         0.0%
Llm Synthesis                                   23061      23.06s        74.3%
--------------------------------------------------------------------------------
TOTAL END-TO-END                                31030      31.03s       100.0%
================================================================================

[Context Statistics]
  Combined context: 42,889 chars (~10,722 tokens)
  KPI block: 0 chars
  RAG block: 42,494 chars
[LLM Invocation]
  Model: us.anthropic.claude-haiku-4-5-20251001-v1:0
  Input tokens: 12,950
  Output tokens: 1,811
  Total tokens: 14,761
  Cost: $0.0220

[Latency Estimates]
  Observed (P50): 31.0s
  Estimated P95: 35.7s  (observed + 15% network variance)
  Estimated P99: 40.3s  (observed + 30% tail latency)

[Bottleneck Analysis]
  LLM synthesis: 74.3% of total latency
  Pipeline overhead: 25.7% of total latency
  → Primary bottleneck: LLM

================================================================================
✓ PROFILING COMPLETE
================================================================================
```

----
----

================================================================================
query = """Compare Amazon, Meta, Google, and Tesla from 2018 to 2022: 
(1) Revenue growth, operating margins, and R&D spending trends
(2) How each company's management discussed AI/ML investments in their MD&A
(3) Regulatory and antitrust risks mentioned in Risk Factors sections
(4) Supply chain vulnerabilities and competitive moats described in their strategic positioning

Show both quantitative metrics and qualitative explanations for how these companies evolved 
their tech platform strategies during this period."""
================================================================================

```python
================================================================================
LATENCY PROFILING - FinRAG Pipeline
================================================================================
Query: Compare Amazon, Meta, Google, and Tesla from 2018 to 2022: 
(1) Revenue growth, ...
================================================================================

[DEBUG] ✓ Found ModelPipeline via file path: D:\JoelDesktop folds_24\NEU FALL2025\MLops IE7374 Project\FinSights\ModelPipeline
[DEBUG] ✓ AWS credentials loaded from aws_credentials.env
[DEBUG] ✓ Found ModelPipeline via file path: D:\JoelDesktop folds_24\NEU FALL2025\MLops IE7374 Project\FinSights\ModelPipeline
[DEBUG] ✓ AWS credentials loaded from aws_credentials.env
[DEBUG] ✓ Local dev detected → LOCAL_CACHE mode
[DEBUG] ✓ Found ModelPipeline via file path: D:\JoelDesktop folds_24\NEU FALL2025\MLops IE7374 Project\FinSights\ModelPipeline
[DEBUG] ✓ AWS credentials loaded from aws_credentials.env
✓ Stage 1: Initialization complete (493ms)
✓ Stage 2: KPI Pipeline complete (924ms)
✓ Stage 3: RAG Pipeline complete (6383ms)
✓ Stage 4: Context assembly complete (0ms)
✓ Stage 5: Prompt formatting complete (0ms)
[DEBUG] Raw length: 25435
[DEBUG] Cleaned length: 25373
[DEBUG] First 200 chars: # Comparative Analysis: Amazon, Meta, Google, and Tesla (2018-2022)

The provided dataset contains substantial limitations that constrain comprehensive analysis across all four companies and requested
✓ Stage 6: LLM synthesis complete (312078ms)

================================================================================
LATENCY BREAKDOWN
================================================================================
Stage                                       Time (ms)    Seconds      % Total
--------------------------------------------------------------------------------
Initialization                                    493       0.49s         0.2%
Kpi Pipeline                                      924       0.92s         0.3%
Rag Pipeline                                     6383       6.38s         2.0%
Context Combine                                     0       0.00s         0.0%
Prompt Format                                       0       0.00s         0.0%
Llm Synthesis                                  312078     312.08s        97.6%
--------------------------------------------------------------------------------
TOTAL END-TO-END                               319877     319.88s       100.0%
================================================================================

[Context Statistics]
  Combined context: 44,444 chars (~11,111 tokens)
  KPI block: 728 chars
  RAG block: 43,046 chars
[LLM Invocation]
  Model: us.anthropic.claude-haiku-4-5-20251001-v1:0
  Input tokens: 14,677
  Output tokens: 5,658
  Total tokens: 20,335
  Cost: $0.0430

[Latency Estimates]
  Observed (P50): 319.9s
  Estimated P95: 367.9s  (observed + 15% network variance)
  Estimated P99: 415.8s  (observed + 30% tail latency)

[Bottleneck Analysis]
  LLM synthesis: 97.6% of total latency
  Pipeline overhead: 2.4% of total latency
  → Primary bottleneck: LLM

================================================================================
✓ PROFILING COMPLETE
================================================================================
```

----
